In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (f1_score, matthews_corrcoef,
                             average_precision_score, roc_auc_score,
                             precision_recall_curve, classification_report)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train_final.csv')
test  = pd.read_csv('../data/test_final.csv')

X_train = train.drop(columns=['isFraud'])
y_train = train['isFraud']
X_test  = test.drop(columns=['isFraud'])
y_test  = test['isFraud']

# Load model đã tune
best_model = xgb.XGBClassifier()
best_model.load_model('../models/xgb_tuned.json')

y_prob = best_model.predict_proba(X_test)[:, 1]
print("Load model OK!")
print(f"Test size: {X_test.shape}")

Load model OK!
Test size: (2000, 192)


In [2]:
thresholds = np.arange(0.1, 0.9, 0.05)
results = []

for thresh in thresholds:
    y_pred = (y_prob >= thresh).astype(int)
    
    f1        = f1_score(y_test, y_pred, zero_division=0)
    precision = (y_pred[y_test==1].sum()) / (y_pred.sum() + 1e-9)
    recall    = (y_pred[y_test==1].sum()) / (y_test.sum())
    
    results.append({
        'threshold': round(thresh, 2),
        'F1':        round(f1, 4),
        'Precision': round(precision, 4),
        'Recall':    round(recall, 4)
    })

df_thresh = pd.DataFrame(results)
print(df_thresh.to_string(index=False))

 threshold     F1  Precision  Recall
      0.10 0.5000     0.7436  0.3766
      0.15 0.4685     0.7647  0.3377
      0.20 0.4860     0.8667  0.3377
      0.25 0.4860     0.8667  0.3377
      0.30 0.4660     0.9231  0.3117
      0.35 0.4660     0.9231  0.3117
      0.40 0.4510     0.9200  0.2987
      0.45 0.4554     0.9583  0.2987
      0.50 0.4554     0.9583  0.2987
      0.55 0.4600     1.0000  0.2987
      0.60 0.4444     1.0000  0.2857
      0.65 0.4124     1.0000  0.2597
      0.70 0.4124     1.0000  0.2597
      0.75 0.3958     1.0000  0.2468
      0.80 0.3958     1.0000  0.2468
      0.85 0.3617     1.0000  0.2208


In [3]:
best_row = df_thresh.loc[df_thresh['F1'].idxmax()]
best_threshold = best_row['threshold']

print(f"✅ Threshold tốt nhất: {best_threshold}")
print(f"   F1:        {best_row['F1']}")
print(f"   Precision: {best_row['Precision']}")
print(f"   Recall:    {best_row['Recall']}")

✅ Threshold tốt nhất: 0.1
   F1:        0.5
   Precision: 0.7436
   Recall:    0.3766


In [4]:
precision_arr, recall_arr, thresh_arr = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(12, 4))

# PR Curve
plt.subplot(1, 2, 1)
plt.plot(recall_arr, precision_arr, color='purple', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'Precision-Recall Curve\nAUPRC = {average_precision_score(y_test, y_prob):.4f}')
plt.grid(True, alpha=0.3)

# F1 vs Threshold
plt.subplot(1, 2, 2)
plt.plot(df_thresh['threshold'], df_thresh['F1'], 
         color='coral', marker='o', label='F1')
plt.plot(df_thresh['threshold'], df_thresh['Precision'], 
         color='steelblue', marker='s', label='Precision')
plt.plot(df_thresh['threshold'], df_thresh['Recall'], 
         color='green', marker='^', label='Recall')
plt.axvline(x=best_threshold, color='red', linestyle='--', 
            label=f'Best threshold={best_threshold}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('F1 / Precision / Recall vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/threshold_tuning.png', dpi=80, bbox_inches='tight')
plt.close()
print("✅ Đã lưu biểu đồ!")

✅ Đã lưu biểu đồ!


In [5]:
# Threshold mặc định 0.5
y_pred_default = (y_prob >= 0.5).astype(int)

# Threshold tốt nhất
y_pred_best = (y_prob >= best_threshold).astype(int)

print(f"{'='*50}")
print(f"SO SÁNH THRESHOLD 0.5 vs {best_threshold}")
print(f"{'='*50}")
print(f"\n--- Threshold = 0.5 ---")
print(classification_report(y_test, y_pred_default, 
      target_names=['Legit','Fraud']))

print(f"\n--- Threshold = {best_threshold} ---")
print(classification_report(y_test, y_pred_best, 
      target_names=['Legit','Fraud']))

# Lưu best threshold
import json
with open('../models/best_threshold.json', 'w') as f:
    json.dump({'threshold': float(best_threshold)}, f)
print(f"✅ Đã lưu best threshold: {best_threshold}")

SO SÁNH THRESHOLD 0.5 vs 0.1

--- Threshold = 0.5 ---
              precision    recall  f1-score   support

       Legit       0.97      1.00      0.99      1923
       Fraud       0.96      0.30      0.46        77

    accuracy                           0.97      2000
   macro avg       0.97      0.65      0.72      2000
weighted avg       0.97      0.97      0.97      2000


--- Threshold = 0.1 ---
              precision    recall  f1-score   support

       Legit       0.98      0.99      0.99      1923
       Fraud       0.74      0.38      0.50        77

    accuracy                           0.97      2000
   macro avg       0.86      0.69      0.74      2000
weighted avg       0.97      0.97      0.97      2000

✅ Đã lưu best threshold: 0.1
